# 02 · Results (all numbers read from `experiments/results/` and `bonus/`)

Nothing is recomputed here except figures. Tuning used the dev set only; the test set was evaluated once, after the `results-frozen` tag.

In [1]:
import csv, json, pathlib, sys
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src import data, pipeline, enhancement, frequency, segmentation
RES, FIG = ROOT / 'experiments/results', ROOT / 'experiments/figures'
DATA = ROOT / 'data/raw/concrete_crack'
L = 1008
PAIRS = {i.stem: (i, g) for i, g in data.list_pairs(DATA)}
def rows(path):
    return list(csv.DictReader(open(path, encoding='utf-8')))
def study(name):
    return [(r['variant'], float(r['iou_mean']), float(r['iou_std'])) for r in rows(RES / f'study_{name}.csv')]
def bar(name, title, fname, color='#2f5bea'):
    s = study(name); fig, ax = plt.subplots(figsize=(8, 0.42 * len(s) + 1))
    ax.barh([v for v, _, _ in s][::-1], [m for _, m, _ in s][::-1], xerr=[sd for _, _, sd in s][::-1], color=color, alpha=.85, capsize=3)
    ax.set_xlabel('mean IoU (error bar = std over images)'); ax.set_title(title)
    fig.tight_layout(); fig.savefig(FIG / fname, dpi=110); plt.close(fig)
    for v, m, sd in sorted(s, key=lambda t: -t[1]): print(f'{v:<50} {m:.4f} +/- {sd:.4f}')

## 1 · Baselines and pipelines: dev (selection) and test (held out, run once)

In [2]:
def mean_std(path):
    v = np.array([float(r['iou']) for r in rows(path)]); return v.mean(), v.std(), v
dev = {v: (m, sd) for v, m, sd in study('final_selection')}
dev['pipeline B'] = next((m, sd) for v, m, sd in study('ablations') if v.startswith('AB4a'))
names = [('trivial baseline', 'baseline_trivial'), ('simple baseline', 'baseline_simple'),
         ('pipeline A', 'pipeline_a'), ('pipeline B', 'pipeline_b'), ('pipeline C (round-2 reference)', 'final')]
table = []
for label, f in names:
    tm, ts, v = mean_std(RES / f'test_{f}.csv')
    table.append((label, *dev[label], tm, ts, v.min(), v.max()))
print(f"{'method':<32}{'dev IoU':>16}{'test IoU':>18}{'test min':>10}{'test max':>10}")
for t in table:
    print(f'{t[0]:<32}{t[1]:>8.4f} +/- {t[2]:.3f}{t[3]:>10.4f} +/- {t[4]:.3f}{t[5]:>10.3f}{t[6]:>10.3f}')
fig, ax = plt.subplots(figsize=(8, 3.6)); x = np.arange(len(table))
ax.bar(x - .2, [t[1] for t in table], .4, yerr=[t[2] for t in table], label='dev (321)', capsize=3, color='#9aa6b6')
ax.bar(x + .2, [t[3] for t in table], .4, yerr=[t[4] for t in table], label='test (137)', capsize=3, color='#2f5bea')
ax.set_xticks(x, ['trivial', 'simple', 'A', 'B', 'C (final)']); ax.set_ylabel('IoU')
ax.legend(); ax.set_title('Baselines and pipelines'); fig.tight_layout()
fig.savefig(FIG / 'fig_pipelines_dev_test.png', dpi=110); plt.close(fig)

method                                   dev IoU          test IoU  test min  test max
trivial baseline                  0.4416 +/- 0.193    0.4486 +/- 0.189     0.017     0.843
simple baseline                   0.0658 +/- 0.144    0.0657 +/- 0.140     0.010     0.816
pipeline A                        0.4582 +/- 0.238    0.4426 +/- 0.243     0.009     0.878
pipeline B                        0.5118 +/- 0.129    0.5134 +/- 0.139     0.187     0.781
pipeline C (round-2 reference)    0.6657 +/- 0.120    0.6695 +/- 0.116     0.330     0.881


In [3]:
final = rows(RES / 'test_final.csv'); triv = {r['name']: float(r['iou']) for r in rows(RES / 'test_baseline_trivial.csv')}
wins = sum(float(r['iou']) > triv[r['name']] for r in final)
print(f'final pipeline beats the trivial baseline on {wins}/{len(final)} test images')
best, worst = max(final, key=lambda r: float(r['iou'])), min(final, key=lambda r: float(r['iou']))
print('best test image', best['name'], round(float(best['iou']), 4), '| worst', worst['name'], round(float(worst['iou']), 4))
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist([float(r['iou']) for r in final], 20, range=(0, 1), alpha=.8, label='final (C)')
ax.hist(list(triv.values()), 20, range=(0, 1), alpha=.6, label='trivial baseline')
ax.set_xlabel('IoU per test image'); ax.legend(); fig.tight_layout()
fig.savefig(FIG / 'fig_test_iou_hist.png', dpi=110); plt.close(fig)

final pipeline beats the trivial baseline on 131/137 test images
best test image 174 0.8808 | worst 249 0.3295


## 2 · Measurement and decision on the test set

In [4]:
cfg = pipeline.load_config(ROOT / 'configs/final_pipeline.yaml'); cut = cfg['severity_cutoffs']
def sev(a):
    return 'minor' if a < cut[0] else ('moderate' if a < cut[1] else 'severe')
ap = np.array([float(r['area_pct_pred']) for r in final]); ag = np.array([float(r['area_pct_gt']) for r in final])
lp = np.array([float(r['length_pred']) for r in final]); lg = np.array([float(r['length_gt']) for r in final])
acc = np.mean([sev(p) == sev(g) for p, g in zip(ap, ag)])
print(f'cracked area %: mean abs error {np.abs(ap - ag).mean():.3f} points (GT mean {ag.mean():.3f})')
print(f'crack length: mean abs error {np.abs(lp - lg).mean():.0f} px (GT mean {lg.mean():.0f} px); relative {np.mean(np.abs(lp - lg) / lg):.1%}')
print(f'severity class agreement with GT-derived class: {acc:.1%} '
      f'(cut-offs {cut} from dev GT tertiles)')
conf = np.zeros((3, 3), int); k = {'minor': 0, 'moderate': 1, 'severe': 2}
for p, g in zip(ap, ag): conf[k[sev(g)], k[sev(p)]] += 1
print('confusion (rows GT, cols predicted: minor/moderate/severe):'); print(conf)
with open(RES / 'measurements_test.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['image', 'area_pct_pred', 'area_pct_gt', 'length_pred', 'length_gt', 'objects_pred', 'severity_pred', 'severity_gt', 'iou'])
    for r in final:
        w.writerow([r['name'], round(float(r['area_pct_pred']), 3), round(float(r['area_pct_gt']), 3), r['length_pred'], r['length_gt'], r['objects_pred'], sev(float(r['area_pct_pred'])), sev(float(r['area_pct_gt'])), round(float(r['iou']), 4)])
fig, ax = plt.subplots(figsize=(4.2, 4)); ax.scatter(ag, ap, s=12)
m = max(ag.max(), ap.max()); ax.plot([0, m], [0, m], 'k--', lw=1)
ax.set_xlabel('GT cracked area %'); ax.set_ylabel('predicted cracked area %')
ax.set_title('Test set measurement'); fig.tight_layout()
fig.savefig(FIG / 'fig_measurement_scatter.png', dpi=110); plt.close(fig)

cracked area %: mean abs error 0.326 points (GT mean 1.412)
crack length: mean abs error 277 px (GT mean 1414 px); relative 18.6%
severity class agreement with GT-derived class: 62.8% (cut-offs [1.0, 1.37] from dev GT tertiles)
confusion (rows GT, cols predicted: minor/moderate/severe):
[[33  5  0]
 [22 15  2]
 [ 1 21 38]]


## 3 · One-factor-at-a-time studies (dev tune set, 60 images)

In [5]:
bar('threshold_method', 'Threshold method (after median + flattening, no morphology)', 'fig_threshold_methods.png')
print()
bar('detectors', 'Edge / line detectors (top 3 % responses)', 'fig_detectors.png')
print()
bar('kernel_size', 'Smoothing filter and kernel size (in Pipeline C, round 2)', 'fig_kernel_size.png', '#0e9384')
print()
bar('cutoff_d0', 'Flattening cutoff D0 (round 2)', 'fig_d0.png', '#7a3cdb')
print()
bar('se_size_shape', 'Closing structuring element (round 2)', 'fig_se.png', '#2b8a3e')
print()
bar('adaptive_params', 'Adaptive threshold block and c (round 2)', 'fig_adaptive.png', '#d9480f')
print()
bar('enhancement_methods', 'Enhancement before the same adaptive threshold', 'fig_enhancement_methods.png', '#0e9384')

otsu                                               0.4885 +/- 0.2227
fixed t=127                                        0.4727 +/- 0.1465
LoG sigma=2 t=2                                    0.4249 +/- 0.1043
fixed t=115                                        0.4207 +/- 0.1244
fixed t=100                                        0.3351 +/- 0.1279
line masks t=60                                    0.3163 +/- 0.0780
adaptive gaussian 31/10                            0.2955 +/- 0.1129
adaptive mean 31/10                                0.2575 +/- 0.1154



prewitt                                            0.2914 +/- 0.0846
sobel                                              0.2912 +/- 0.0845
roberts                                            0.2840 +/- 0.0835
log                                                0.2560 +/- 0.0538
lines                                              0.2343 +/- 0.0672
laplacian                                          0.2046 +/- 0.0559

gaussian k=5                                       0.6790 +/- 0.1025
mean k=3                                           0.6714 +/- 0.1011
gaussian k=7                                       0.6709 +/- 0.1143
mean k=5                                           0.6582 +/- 0.1166
gaussian k=3                                       0.6557 +/- 0.1001
median k=3                                         0.6143 +/- 0.1069
mean k=7                                           0.5911 +/- 0.1505
median k=5                                         0.5735 +/- 0.1500
no smoothing                     

disk 7                                             0.6796 +/- 0.1083
disk 5                                             0.6767 +/- 0.1090
square 5                                           0.6734 +/- 0.1094
square 7                                           0.6724 +/- 0.1072
disk 3                                             0.6709 +/- 0.1143
square 3                                           0.6685 +/- 0.1151
no closing                                         0.6683 +/- 0.1150



block=31                                           0.6709 +/- 0.1143
block=61                                           0.6636 +/- 0.1108
c=10                                               0.6398 +/- 0.1122
c=20                                               0.6340 +/- 0.1343
method=gaussian                                    0.5819 +/- 0.1233
block=15                                           0.5732 +/- 0.1216
c=25                                               0.5714 +/- 0.1618

log                                                0.4780 +/- 0.0918
mean k=5                                           0.3915 +/- 0.1323
median k=5                                         0.3558 +/- 0.1277
gaussian k=5                                       0.3215 +/- 0.1289
gamma 0.5                                          0.3202 +/- 0.1152
none                                               0.1658 +/- 0.0904
gamma 2.0                                          0.1095 +/- 0.0812
gradient sharpen                 

## 4 · Enhancement: images and histograms (one dev image)

In [6]:
name = data.load_split(ROOT / 'data/split.json')['dev'][0]
g = data.load_image(PAIRS[name][0], size=L)
methods = [('original', g), ('log', enhancement.log_transform(g)), ('gamma 0.5', enhancement.gamma_transform(g, .5)),
           ('contrast stretch', enhancement.contrast_stretch(g)), ('hist. equalization', enhancement.hist_equalize(g)),
           ('Gaussian 5', enhancement.gaussian_filter(g, 5)), ('Laplacian sharpen', enhancement.laplacian_sharpen(g))]
fig, ax = plt.subplots(2, len(methods), figsize=(3 * len(methods), 5.5))
for j, (t, im) in enumerate(methods):
    ax[0, j].imshow(im, cmap='gray', vmin=0, vmax=255); ax[0, j].set_title(t); ax[0, j].axis('off')
    ax[1, j].bar(range(256), enhancement.histogram(im), width=1)
fig.tight_layout(); fig.savefig(FIG / 'fig_enhancement_images.png', dpi=80); plt.close(fig)

## 5 · Restoration (MEM3): PSNR by noise type, level and filter

In [7]:
r = rows(RES / 'study_restoration_psnr.csv')
noises = list(dict.fromkeys((x['noise'], x['level']) for x in r)); filters = list(dict.fromkeys(x['filter'] for x in r))
M = np.array([[next(float(x['psnr_db']) for x in r if (x['noise'], x['level']) == n and x['filter'] == f) for n in noises] for f in filters])
fig, ax = plt.subplots(figsize=(10, 7)); im = ax.imshow(M, cmap='viridis')
ax.set_xticks(range(len(noises)), [f'{a} {b}' for a, b in noises], rotation=40, ha='right')
ax.set_yticks(range(len(filters)), filters)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f'{M[i, j]:.1f}', ha='center', va='center', fontsize=7, color='w')
fig.colorbar(im, label='PSNR (dB)'); fig.tight_layout()
fig.savefig(FIG / 'fig_restoration_psnr.png', dpi=100); plt.close(fig)
for j, n in enumerate(noises):
    i = int(np.argmax(M[:, j])); print(f'{n[0]:12} {n[1]:<5} noisy {M[0, j]:.2f} dB -> best {filters[i]} {M[i, j]:.2f} dB')

gaussian     10    noisy 28.13 dB -> best butterworth LP D0=400 34.32 dB
gaussian     20    noisy 22.14 dB -> best gaussian LP D0=200 32.06 dB
gaussian     30    noisy 18.73 dB -> best gaussian LP D0=100 30.90 dB
salt_pepper  0.02  noisy 22.29 dB -> best median 37.10 dB
salt_pepper  0.05  noisy 18.32 dB -> best median 36.87 dB
salt_pepper  0.1   noisy 15.30 dB -> best median 36.08 dB
speckle      0.1   noisy 23.34 dB -> best gaussian LP D0=200 32.58 dB
speckle      0.2   noisy 17.64 dB -> best gaussian LP D0=100 30.55 dB
speckle      0.3   noisy 14.54 dB -> best butterworth LP D0=100 28.90 dB


## 6 · Frequency domain (MEM3): spectra, cutoff D0, ringing

In [8]:
img = data.add_noise(g, 'gaussian', 20, 0); shape = (2 * img.shape[0], 2 * img.shape[1])
fig, ax = plt.subplots(3, 4, figsize=(16, 11))
for j, d0 in enumerate((50, 100, 200, 400)):
    for i, kind in enumerate(('ideal', 'butterworth', 'gaussian')):
        h = {'ideal': frequency.ideal_lowpass(shape, d0), 'butterworth': frequency.butterworth_lowpass(shape, d0),
             'gaussian': frequency.gaussian_lowpass(shape, d0)}[kind]
        ax[i, j].imshow(frequency.apply_filter(img, h)[300:500, 300:550], cmap='gray', vmin=0, vmax=255)
        ax[i, j].set_title(f'{kind} LP, D0={d0}'); ax[i, j].axis('off')
fig.tight_layout(); fig.savefig(FIG / 'fig_frequency_cutoffs.png', dpi=70); plt.close(fig)
h = frequency.gaussian_lowpass(shape, 100); out = frequency.apply_filter(img, h)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (t, im) in zip(ax, [('noisy input (sigma 20)', img), ('spectrum before', frequency.centered_spectrum(img)),
                           ('spectrum after Gaussian LP D0=100', frequency.centered_spectrum(out)), ('filtered', out)]):
    a.imshow(im, cmap='gray' if 'spectrum' not in t else 'magma'); a.set_title(t); a.axis('off')
fig.tight_layout(); fig.savefig(FIG / 'fig_frequency_spectra.png', dpi=80); plt.close(fig)

## 7 · Ablations and robustness (all 321 dev images)

In [9]:
bar('ablations', 'Ablations of the final pipeline', 'fig_ablations.png', '#7a3cdb')
rb = study('robustness'); clean = next(m for v, m, _ in rb if v == 'clean')
print(f'\nclean: {clean:.4f}')
levels = [10, 20, 30]; means = []
for s in levels:
    v = np.array([m for name, m, _ in rb if f'sigma={s} ' in name])
    means.append((v.mean(), v.std())); print(f'Gaussian sigma={s}: IoU {v.mean():.4f} +/- {v.std():.4f} over seeds 0/1/2')
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.errorbar([0] + levels, [clean] + [m for m, _ in means], yerr=[0] + [s for _, s in means], marker='o', capsize=4)
ax.set_xlabel('added Gaussian noise sigma'); ax.set_ylabel('mean IoU (dev)'); fig.tight_layout()
fig.savefig(FIG / 'fig_robustness.png', dpi=110); plt.close(fig)

final pipeline                                     0.6657 +/- 0.1201
AB6 no illumination flattening                     0.6626 +/- 0.1193
AB5 no edge linking                                0.6620 +/- 0.1207
AB2 frequency low-pass instead of spatial smoothing 0.6486 +/- 0.1277
AB4b opening then closing (order)                  0.6410 +/- 0.1388
AB3 global Otsu instead of adaptive threshold      0.5681 +/- 0.2315
AB4a no morphology / object selection              0.5118 +/- 0.1291
AB1 no enhancement (no smoothing, no flattening)   0.5100 +/- 0.1710

clean: 0.6657
Gaussian sigma=10: IoU 0.6618 +/- 0.0005 over seeds 0/1/2
Gaussian sigma=20: IoU 0.6498 +/- 0.0007 over seeds 0/1/2
Gaussian sigma=30: IoU 0.6223 +/- 0.0007 over seeds 0/1/2


## 8 · Failure analysis: 20 worst test images
Panels with every stage: `experiments/figures/failures/`. Categories are assigned by a written rule on the measurements, then checked visually on the panels.

In [10]:
worst = sorted(final, key=lambda r: float(r['iou']))[:20]
def category(r):
    ap, ag, objs = float(r['area_pct_pred']), float(r['area_pct_gt']), int(r['objects_pred'])
    if ag >= 1.5 and ap < 0.6 * ag:
        return 'wide crack / spalling under-segmented'
    if objs >= 15:
        return 'rough texture / pits -> false positives'
    if ap < 0.8 * ag:
        return 'faint thin branches missed'
    return 'boundary width mismatch with GT'
cats = {}
for r in worst:
    cats.setdefault(category(r), []).append(r['name'])
for c, n in sorted(cats.items(), key=lambda kv: -len(kv[1])):
    print(f'{c:<42} {len(n):>2}: {", ".join(n)}')
with open(RES / 'failure_analysis.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['image', 'iou', 'area_pct_pred', 'area_pct_gt', 'objects_pred', 'category'])
    for r in worst:
        w.writerow([r['name'], round(float(r['iou']), 4), round(float(r['area_pct_pred']), 3), round(float(r['area_pct_gt']), 3), r['objects_pred'], category(r)])
print('\nfixes tested on the DEV set (frozen final not changed):')
for v, m, sd in study('failure_fixes'): print(f'  {v:<48} {m:.4f}')

rough texture / pits -> false positives     9: 079, 273, 078, 051, 425, 572, 052, 286, 188
faint thin branches missed                  6: 278, 297, 534, 281, 535, 204
wide crack / spalling under-segmented       4: 249, 293, 074, 251
boundary width mismatch with GT             1: 578

fixes tested on the DEV set (frozen final not changed):
  final pipeline                                   0.6657
  fix wide cracks: adaptive OR Otsu                0.6021
  fix texture: min_area 120, max_circularity 0.3   0.6609


## 9 · Bonus results

In [11]:
for name in ('compression/results.csv', 'generalization/results.csv'):
    print(name, len(rows(ROOT / 'bonus' / name)), 'rows')
comp = rows(ROOT / 'bonus/compression/results.csv')
for kind in ('mask / RLE', 'gray / Huffman'):
    v = [float(r['ratio']) for r in comp if r['data'] == kind]
    print(f'{kind}: mean ratio {np.mean(v):.2f}, all exact: {all(r["exact"] == "True" for r in comp if r["data"] == kind)}')
gen = rows(ROOT / 'bonus/generalization/results.csv')
for k in ('iou_final', 'iou_trivial'):
    v = np.array([float(r[k]) for r in gen]); print(f'DeepCrack {k}: {v.mean():.4f} +/- {v.std():.4f}')
for r in rows(ROOT / 'bonus/optimization/timings.csv'):
    if 'TOTAL' in r['stage']: print('speed', r['label'], r['seconds_per_image'], 's/img')

compression/results.csv 20 rows
generalization/results.csv 237 rows
mask / RLE: mean ratio 440.74, all exact: True
gray / Huffman: mean ratio 1.36, all exact: True
DeepCrack iou_final: 0.5199 +/- 0.2008
DeepCrack iou_trivial: 0.3462 +/- 0.2626
speed before 0.9223 s/img
speed after 0.5016 s/img
